# Chapter 3: Probability — Hands-On Jupyter Notebook
**Haydar Kılıç | Artificial Intelligence Engineering**

This notebook demonstrates the theoretical concepts from Chapter 3 through hands-on Python examples.

---
**Contents:**
1. Random Processes and the Law of Large Numbers
2. Probability Rules (Addition, Multiplication, Complement)
3. Conditional Probability and Independence
4. Bayes' Theorem and Tree Diagrams
5. Sampling With / Without Replacement
6. Random Variables: Expected Value and Variance
7. Continuous Distributions

In [ ]:
# Required libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import pandas as pd
from fractions import Fraction
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(42)
print('Libraries loaded.')

---
## 1. Random Processes and the Law of Large Numbers

**Definition:** The Law of Large Numbers (LLN) states that as the number of observations increases, the observed proportion converges to the true probability:

$$\hat{p}_n \xrightarrow{n \to \infty} p$$

**Gambler's Fallacy:** A coin does not need to "compensate" for past outcomes. Each flip is independent of all others.

In [ ]:
# ── Law of Large Numbers Simulation ──
n_flips = 5000
flips = np.random.choice([0, 1], size=n_flips)  # 1 = Heads
cumulative_rate = np.cumsum(flips) / np.arange(1, n_flips + 1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(cumulative_rate, color='steelblue', lw=1.5, label='Observed heads proportion')
ax.axhline(0.5, color='tomato', lw=2, linestyle='--', label='True probability p = 0.5')
ax.fill_between(range(n_flips), cumulative_rate, 0.5, alpha=0.15, color='steelblue')
ax.set_xlabel('Number of flips (n)')
ax.set_ylabel('Proportion of heads')
ax.set_title('Law of Large Numbers — Coin Flip Simulation')
ax.legend()
ax.set_ylim(0.2, 0.8)
plt.tight_layout()
plt.show()

print(f'Heads proportion after 10 flips  : {cumulative_rate[9]:.3f}')
print(f'Heads proportion after 100 flips : {cumulative_rate[99]:.3f}')
print(f'Heads proportion after 5000 flips: {cumulative_rate[-1]:.3f}')

In [ ]:
# ── Gambler's Fallacy Demonstration ──
# 10 heads in a row — does the next flip have to be tails?
# Each flip is independent; the probability is still 0.5!

n_trials = 100_000
# Simulate the flip that follows 10 heads
next_flip = np.random.choice([0, 1], size=n_trials)  # 1=Heads, 0=Tails
heads_rate = next_flip.mean()

print('=== Gambler\'s Fallacy Test ===')
print(f'Simulation of {n_trials} trials after 10 consecutive heads:')
print(f'  Heads rate : {heads_rate:.4f}  (Expected: 0.5000)')
print(f'  Tails rate : {1-heads_rate:.4f}  (Expected: 0.5000)')
print()
print('Result: The coin has no memory. P(Heads) = 0.5 on every flip.')

---
## 2. Probability Rules

### 2.1 General Addition Rule

$$P(A \text{ or } B) = P(A) + P(B) - P(A \text{ and } B)$$

For disjoint (mutually exclusive) events $P(A \text{ and } B) = 0$, so:

$$P(A \text{ or } B) = P(A) + P(B)$$

### 2.2 Complement Rule
$$P(A^c) = 1 - P(A)$$

In [ ]:
# ── Deck of Cards: Jack or Red Card ──
# P(jack or red) = P(jack) + P(red) − P(jack and red)

total_cards = 52
p_jack        = Fraction(4, total_cards)   # 4 jacks
p_red         = Fraction(26, total_cards)  # 26 red cards
p_jack_red    = Fraction(2, total_cards)   # 2 red jacks

p_jack_or_red = p_jack + p_red - p_jack_red

print('=== Jack or Red Card ===')
print(f'P(Jack)              = {p_jack} = {float(p_jack):.4f}')
print(f'P(Red)               = {p_red} = {float(p_red):.4f}')
print(f'P(Jack and Red)      = {p_jack_red} = {float(p_jack_red):.4f}')
print(f'P(Jack or Red)       = {p_jack} + {p_red} - {p_jack_red} = {p_jack_or_red} ≈ {float(p_jack_or_red):.4f}')

In [ ]:
# ── Complement Rule: At Least 1 Uninsured Texan ──
# P(at least 1 uninsured) = 1 − P(none uninsured)

p_uninsured = 0.255
n = 5

p_none_uninsured = (1 - p_uninsured) ** n
p_at_least_one   = 1 - p_none_uninsured

print('=== 5 Texas Residents: At Least 1 Uninsured ===')
print(f'P(uninsured)              = {p_uninsured}')
print(f'P(insured)                = {1 - p_uninsured}')
print(f'P(none uninsured)         = {1-p_uninsured}^5 = {p_none_uninsured:.4f}')
print(f'P(at least 1 uninsured)   = 1 − {p_none_uninsured:.4f} = {p_at_least_one:.4f}')

# Visualization: probability as a function of n
ns    = np.arange(1, 21)
probs = 1 - (1 - p_uninsured) ** ns

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(ns, probs, color='steelblue', edgecolor='white', alpha=0.85)
ax.axhline(0.95, color='tomato', linestyle='--', lw=1.5, label='0.95 threshold')
ax.set_xlabel('Sample size (n)')
ax.set_ylabel('P(at least 1 uninsured)')
ax.set_title('Complement Rule — Probability by Sample Size')
ax.legend()
plt.tight_layout()
plt.show()

n_95 = next(i for i, p in enumerate(probs, 1) if p >= 0.95)
print(f'\nMinimum n for P(at least 1 uninsured) ≥ 0.95: {n_95}')

---
## 3. Conditional Probability and Independence

$$P(A|B) = \frac{P(A \text{ and } B)}{P(B)}$$

**Independence test:** A and B are independent if $P(A|B) = P(A)$.

In [ ]:
# ── Relapse Study: Conditional Probability ──
data = pd.DataFrame({
    'Treatment': ['Desipramine', 'Desipramine', 'Lithium', 'Lithium', 'Placebo', 'Placebo'],
    'Outcome'  : ['Relapse', 'No Relapse', 'Relapse', 'No Relapse', 'Relapse', 'No Relapse'],
    'Count'    : [10, 14, 18, 6, 20, 4]
})

table = data.pivot(index='Treatment', columns='Outcome', values='Count')
table['Total'] = table.sum(axis=1)
table.loc['Total'] = table.sum()
print('=== Relapse Study Contingency Table ===')
print(table)

total = 72
print()
# Marginal probability
p_relapse = 48/72
print(f'P(Relapse)                    = 48/72 = {p_relapse:.4f}')

# Joint probability
p_des_and_relapse = 10/72
print(f'P(Desipramine and Relapse)    = 10/72 = {p_des_and_relapse:.4f}')

# Conditional probabilities
p_relapse_des = 10/24
p_relapse_lit = 18/24
p_relapse_pla = 20/24
print(f'\nP(Relapse | Desipramine)      = 10/24 = {p_relapse_des:.4f}')
print(f'P(Relapse | Lithium)          = 18/24 = {p_relapse_lit:.4f}')
print(f'P(Relapse | Placebo)          = 20/24 = {p_relapse_pla:.4f}')

In [ ]:
# ── Visualization: Conditional Relapse Rates by Treatment ──
treatments      = ['Desipramine', 'Lithium', 'Placebo']
relapse_rate    = [10/24, 18/24, 20/24]
no_relapse_rate = [14/24,  6/24,  4/24]

x = np.arange(len(treatments))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, relapse_rate,    w, label='Relapse',    color='tomato',    alpha=0.85)
b2 = ax.bar(x + w/2, no_relapse_rate, w, label='No Relapse', color='steelblue', alpha=0.85)

ax.axhline(48/72, color='gray', linestyle='--', lw=1.5,
           label=f'Marginal P(Relapse) = {48/72:.2f}')
ax.set_xticks(x)
ax.set_xticklabels(treatments)
ax.set_ylabel('Conditional Probability')
ax.set_title('Conditional Relapse Probabilities by Treatment')
ax.legend()

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Independence Test: Gun Ownership & Race/Ethnicity ──
# Independent if P(A|B) = P(A)

p_general  = 0.58
p_white    = 0.67
p_black    = 0.28
p_hispanic = 0.64

print('=== Gun Ownership Opinion and Race/Ethnicity — Independence Test ===')
print(f'P(Protects) [overall]           = {p_general}')
print(f'P(Protects | White)             = {p_white}')
print(f'P(Protects | Black)             = {p_black}')
print(f'P(Protects | Hispanic)          = {p_hispanic}')
print()
print('Conditional probabilities differ from each other and from the overall P(Protects).')
print('→ Opinion and race/ethnicity are likely DEPENDENT.')

---
## 4. Bayes' Theorem and Probability Trees

$$P(A_1|B) = \frac{P(B|A_1)\,P(A_1)}{P(B|A_1)\,P(A_1) + P(B|A_2)\,P(A_2) + \cdots + P(B|A_k)\,P(A_k)}$$

In [ ]:
# ── Breast Cancer Screening ──
def bayes_update(p_cancer, sensitivity=0.78, false_positive_rate=0.10):
    """Computes the probability of cancer given a positive mammogram."""
    p_no_cancer       = 1 - p_cancer
    # P(+ | cancer) × P(cancer)
    p_pos_cancer      = sensitivity * p_cancer
    # P(+ | no cancer) × P(no cancer)
    p_pos_no_cancer   = false_positive_rate * p_no_cancer
    # Bayes' formula
    p_cancer_given_pos = p_pos_cancer / (p_pos_cancer + p_pos_no_cancer)
    return p_cancer_given_pos, p_pos_cancer, p_pos_no_cancer

# First test
p_prior = 0.017
p_posterior, p_pos_cancer, p_pos_no_cancer = bayes_update(p_prior)

print('=== Mammogram — Test 1 ===')
print(f'Prior probability P(Cancer)           = {p_prior}')
print(f'P(+ | Cancer) × P(Cancer)             = {0.78}×{p_prior} = {p_pos_cancer:.4f}')
print(f'P(+ | No Cancer) × P(No Cancer)       = {0.10}×{1-p_prior:.3f} = {p_pos_no_cancer:.4f}')
print(f'P(Cancer | +)                         = {p_pos_cancer:.4f}/{p_pos_cancer+p_pos_no_cancer:.4f} = {p_posterior:.4f}')

# Second test (with updated prior)
p_prior2 = p_posterior
p_posterior2, p_pos2, p_no2 = bayes_update(p_prior2)
print()
print('=== Mammogram — Test 2 (prior updated with positive result) ===')
print(f'Updated prior P(Cancer)               = {p_prior2:.4f}')
print(f'P(Cancer | +) [Test 2]                = {p_posterior2:.4f}')

In [ ]:
# ── Visualization of Bayesian Updating ──
priors     = np.linspace(0.001, 0.5, 200)
posteriors = [bayes_update(p)[0] for p in priors]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(priors, posteriors, color='steelblue', lw=2)
ax.plot([0, 0.5], [0, 0.5], 'gray', linestyle=':', lw=1.5, label='Prior = Posterior (independent test)')

# Mark the two tests
ax.scatter([0.017, p_posterior], [p_posterior, p_posterior2],
           s=100, zorder=5, color='tomato',
           label=f'Test 1: prior={0.017}, posterior={p_posterior:.3f}\n'
                 f'Test 2: prior={p_posterior:.3f}, posterior={p_posterior2:.3f}')

ax.set_xlabel('Prior Probability P(Cancer)')
ax.set_ylabel('Posterior Probability P(Cancer | +)')
ax.set_title('Bayesian Updating — Mammogram Example')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── SIR Model: Test Effectiveness ──
# Population distribution: Susceptible 60%, Infected 10%, Recovered 30%
# Test accuracy: S → 95% negative, I → 99% positive, R → 65% negative

p_s, p_i, p_r = 0.60, 0.10, 0.30

# P(positive) for each group
p_pos_s = 0.05   # susceptible  → false positive
p_pos_i = 0.99   # infected     → true positive
p_pos_r = 0.35   # recovered    → false positive

# Joint probabilities
p_s_pos     = p_s * p_pos_s
p_i_pos     = p_i * p_pos_i
p_r_pos     = p_r * p_pos_r
p_pos_total = p_s_pos + p_i_pos + p_r_pos

# Bayes
p_inf_given_pos = p_i_pos / p_pos_total

print('=== SIR Model — Bayes\' Theorem ===')
print(f'P(Susceptible) × P(+ | Susceptible)  = {p_s}×{p_pos_s} = {p_s_pos:.4f}')
print(f'P(Infected)    × P(+ | Infected)     = {p_i}×{p_pos_i} = {p_i_pos:.4f}')
print(f'P(Recovered)   × P(+ | Recovered)    = {p_r}×{p_pos_r} = {p_r_pos:.4f}')
print(f'P(+) [total]                         = {p_pos_total:.4f}')
print(f'P(Infected | +)                      = {p_i_pos:.4f}/{p_pos_total:.4f} = {p_inf_given_pos:.4f}')

# Pie charts
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

axs[0].pie([p_s, p_i, p_r], labels=['Susceptible', 'Infected', 'Recovered'],
           autopct='%1.0f%%', colors=['#5B9BD5', '#ED7D31', '#70AD47'],
           startangle=90)
axs[0].set_title('Population Distribution')

axs[1].pie([p_s_pos, p_i_pos, p_r_pos],
           labels=[f'Susceptible+\n({p_s_pos:.3f})',
                   f'Infected+\n({p_i_pos:.3f})',
                   f'Recovered+\n({p_r_pos:.3f})'],
           autopct='%1.1f%%', colors=['#5B9BD5', '#ED7D31', '#70AD47'],
           startangle=90)
axs[1].set_title(f'Positive Testers\nP(Infected|+) = {p_inf_given_pos:.3f}')

plt.suptitle('SIR Model — Bayesian Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Sampling With / Without Replacement

| | With Replacement | Without Replacement |
|---|---|---|
| Bag composition | Unchanged | Changes |
| Draws | Independent | Dependent |

In [ ]:
# ── Chip Bag Simulation ──
# 5 red, 3 blue, 2 orange
bag = ['R']*5 + ['B']*3 + ['O']*2
print(f'Bag: {bag}  (total {len(bag)} chips)')

n_trials = 100_000

# With replacement: 2 blue
wr_count = 0
for _ in range(n_trials):
    c1 = np.random.choice(bag)
    c2 = np.random.choice(bag)       # bag unchanged
    if c1 == 'B' and c2 == 'B':
        wr_count += 1

# Without replacement: 2 blue
wor_count = 0
for _ in range(n_trials):
    remaining = bag.copy()
    c1 = np.random.choice(remaining)
    remaining.remove(c1)
    c2 = np.random.choice(remaining)  # bag changed
    if c1 == 'B' and c2 == 'B':
        wor_count += 1

print()
print('=== Drawing 2 Blue Chips in a Row ===')
print(f'Theoretical (WR)  : P(B)×P(B)     = 0.3×0.3 = 0.09')
print(f'Simulated  (WR)   :                           = {wr_count/n_trials:.4f}')
print()
print(f'Theoretical (WOR) : P(B)×P(B|B)   = 0.3×0.222 = {0.3*2/9:.4f}')
print(f'Simulated  (WOR)  :                            = {wor_count/n_trials:.4f}')

In [ ]:
# ── Playing Cards: Ace then 3 (Without Replacement) ──
p_ace          = Fraction(4, 52)
p_three_after_ace = Fraction(4, 51)   # remaining after ace is drawn
p_ace_then_3   = p_ace * p_three_after_ace

print('=== Ace then 3 (Without Replacement) ===')
print(f'P(Ace)                    = 4/52 = {float(p_ace):.4f}')
print(f'P(3 | Ace drawn)          = 4/51 = {float(p_three_after_ace):.4f}')
print(f'P(Ace then 3)             = {p_ace_then_3} ≈ {float(p_ace_then_3):.4f}')

---
## 6. Random Variables: Expected Value and Variance

$$E(X) = \mu = \sum_{i=1}^{k} x_i \, P(X = x_i)$$

$$\text{Var}(X) = \sigma^2 = \sum_{i=1}^{k} (x_i - E(X))^2 \, P(X = x_i)$$

In [ ]:
# ── Card Game: Expected Value and Variance ──
# Heart (non-ace) → $1 | Ace → $5 | Queen of Spades → $10 | Other → $0

winnings = np.array([1,    5,    10,   0  ])
probs    = np.array([12/52, 4/52, 1/52, 35/52])
labels   = ['Heart\n(non-ace)', 'Ace', 'Queen\nof Spades', 'Other']

# Expected value
E_X = np.sum(winnings * probs)

# Variance
Var_X = np.sum((winnings - E_X)**2 * probs)
SD_X  = np.sqrt(Var_X)

# Table
df = pd.DataFrame({'Outcome':        labels,
                   'Winnings ($)':    winnings,
                   'P(X)':           [f'{p:.4f}' for p in probs],
                   'X·P(X)':         winnings * probs,
                   '(X-E[X])²·P(X)': (winnings - E_X)**2 * probs})
print(df.to_string(index=False))
print(f'\nE(X)   = {E_X:.4f}')
print(f'Var(X) = {Var_X:.4f}')
print(f'SD(X)  = {SD_X:.4f}')

In [ ]:
# ── Probability Distribution Plot ──
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue', 'steelblue', 'steelblue', 'lightgray']
ax.bar(winnings, probs, width=0.6, color=colors, edgecolor='white')
ax.axvline(E_X, color='tomato', lw=2, linestyle='--', label=f'E(X) = {E_X:.2f}')
ax.set_xlabel('Winnings ($)')
ax.set_ylabel('Probability P(X)')
ax.set_title('Card Game — Probability Distribution of Winnings')
ax.set_xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Linear Combinations: Homework Duration ──
# E(5P + 4C) = 5·E(P) + 4·E(C)
# Var(5P + 4C) = 5²·Var(P) + 4²·Var(C)  [independence assumption]

E_physics  = 10   # min
E_chem     = 15   # min
SD_physics = 1.5  # min
SD_chem    = 2.0  # min

n_physics, n_chem = 5, 4

E_total   = n_physics * E_physics + n_chem * E_chem
Var_total = n_physics * SD_physics**2 + n_chem * SD_chem**2
SD_total  = np.sqrt(Var_total)

print('=== Weekly Homework Duration ===')
print(f'E(5·Physics + 4·Chem) = 5×{E_physics} + 4×{E_chem} = {E_total} minutes')
print(f'Var(5·Physics + 4·Chem) = 5×{SD_physics}² + 4×{SD_chem}² = {Var_total}')
print(f'SD(Total) = √{Var_total} = {SD_total:.4f} minutes')

print()
print('=== Difference Between X+X and 2X ===')
E_XX   = 2 * E_physics
E_2X   = 2 * E_physics
Var_XX = 2 * SD_physics**2
Var_2X = 4 * SD_physics**2
print(f'E(X+X) = {E_XX}  =  E(2X) = {E_2X}   → Expected values are equal')
print(f'Var(X+X) = 2×{SD_physics}² = {Var_XX}  ≠  Var(2X) = 4×{SD_physics}² = {Var_2X}   → Variances are DIFFERENT!')

In [ ]:
# ── Casino Game: Fair Game Test ──
# Entry fee $5. If the first card is red, a second card is drawn.
# If it is the ace of spades, win $500; otherwise $0.

entry_fee = 5
p_red_then_ace_spades = (26/52) * (1/51)   # draw red then ace of spades
p_other               = 1 - p_red_then_ace_spades

profit_win  = 500 - entry_fee
profit_lose = 0   - entry_fee

E_profit = profit_win * p_red_then_ace_spades + profit_lose * p_other

print('=== Casino Game ===')
print(f'P(Red and Ace of Spades) = 26/52 × 1/51 = {p_red_then_ace_spades:.4f}')
print(f'Profit if win            = 500 - 5 = {profit_win}')
print(f'Profit if lose           = 0 - 5   = {profit_lose}')
print(f'E(Profit)                = {profit_win}×{p_red_then_ace_spades:.4f} + ({profit_lose})×{p_other:.4f}')
print(f'                         = {E_profit:.4f}  (≈ {abs(E_profit)*100:.1f} cents lost per play)')
print()
print('This game is not in the player\'s favor — there is an expected loss.')
print('For a fair game: E(Profit) = 0  →  Entry fee ≈ {:.2f}'.format(
    500 * p_red_then_ace_spades))

---
## 7. Continuous Distributions

For continuous random variables, probability is measured by the **area under the curve** of the probability density function:

$$P(a \leq X \leq b) = \int_a^b f(x)\,dx$$

The probability of any single point is always zero: $P(X = c) = 0$

In [ ]:
# ── Continuous Distribution: Height of US Adults ──
# Normal distribution: μ = 170 cm, σ = 10 cm (approximate)

mu, sigma = 170, 10
x   = np.linspace(130, 210, 500)
pdf = stats.norm.pdf(x, mu, sigma)

# P(180 ≤ X ≤ 185)
a, b     = 180, 185
p_interval = stats.norm.cdf(b, mu, sigma) - stats.norm.cdf(a, mu, sigma)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax in axes:
    ax.plot(x, pdf, 'steelblue', lw=2.5)
    ax.fill_between(x, pdf, alpha=0.08, color='steelblue')
    ax.set_xlabel('Height (cm)')
    ax.set_ylabel('Probability Density')

# Left: 180–185 interval
xf = np.linspace(a, b, 300)
axes[0].fill_between(xf, stats.norm.pdf(xf, mu, sigma),
                     color='steelblue', alpha=0.7)
axes[0].set_title(f'P(180 ≤ X ≤ 185) = {p_interval:.4f}')

# Right: Exactly 180 cm — probability is 0
axes[1].axvline(180, color='tomato', lw=2.5, label='X = 180')
axes[1].set_title('P(X = 180) = 0  (single point, no area)')
axes[1].legend()

plt.suptitle('Continuous Distribution — Height Example', fontsize=14)
plt.tight_layout()
plt.show()

print(f'P(180 ≤ Height ≤ 185) = {p_interval:.4f}')
print(f'P(Height = 180)        = 0  (by definition)')

In [ ]:
# ── Probabilities for Different Intervals ──
intervals = [(155, 165), (165, 175), (175, 185), (185, 195)]
colors    = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x, pdf, 'black', lw=2)

for (a_, b_), color in zip(intervals, colors):
    xf = np.linspace(a_, b_, 200)
    p  = stats.norm.cdf(b_, mu, sigma) - stats.norm.cdf(a_, mu, sigma)
    ax.fill_between(xf, stats.norm.pdf(xf, mu, sigma),
                    color=color, alpha=0.55,
                    label=f'P({a_}–{b_}) = {p:.3f}')

ax.set_xlabel('Height (cm)')
ax.set_ylabel('Density')
ax.set_title('Normal Distribution — Probabilities for Different Intervals')
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary and Formulas

| Rule | Formula |
|---|---|
| General Addition | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Complement | $P(A^c) = 1 - P(A)$ |
| Conditional Probability | $P(A|B) = P(A \cap B)/P(B)$ |
| Independent Events | $P(A \cap B) = P(A) \times P(B)$ |
| General Multiplication | $P(A \cap B) = P(A|B) \times P(B)$ |
| Bayes' Theorem | $P(A_1|B) = \frac{P(B|A_1)P(A_1)}{\sum_i P(B|A_i)P(A_i)}$ |
| Expected Value | $E(X) = \sum x_i P(X=x_i)$ |
| Variance | $\text{Var}(X) = \sum (x_i - E(X))^2 P(X=x_i)$ |
| Linear Combination | $E(aX+bY) = aE(X)+bE(Y)$ |
| Indep. Variance | $\text{Var}(aX+bY) = a^2\text{Var}(X)+b^2\text{Var}(Y)$ |